# Treino da LLM local na GPU (Google Colab + VSCode)

Diagnostico de logs da integracao Sienge <-> Hilti com **Qwen2.5-0.5B-Instruct + LoRA**.

**Antes de rodar:** no VSCode, clique em *Select Kernel* -> *Colab* -> *New Colab Server* -> **GPU T4**.
(No navegador: *Ambiente de execucao -> Alterar o tipo -> GPU T4*.)

## 0. GPU instalada e funcionando (print para a entrega)

In [ ]:
!nvidia-smi

## 1. Baixar o projeto

In [ ]:
REPO = "https://github.com/victoreid/llm-diagnostico-logs.git"

import os
os.chdir("/content")
if not os.path.exists("llm-diagnostico-logs"):
    !git clone {REPO} llm-diagnostico-logs
%cd /content/llm-diagnostico-logs
!git pull
!ls data

## 2. Instalar dependencias (o PyTorch com CUDA ja vem no Colab)

In [ ]:
# torchao do Colab e incompativel com o peft (e nao e usado neste projeto)
!pip uninstall -y -q torchao
!pip install -q -r requirements-colab.txt

## 3. Conferir PyTorch + CUDA + GPU

In [ ]:
!python src/verificar_gpu.py

## 4. Avaliar o modelo ANTES do treino (baseline)
O modelo base nunca viu esses logs: serve de comparacao.

In [ ]:
!python src/avaliar.py --nome base

## 5. Treinar e avaliar com diferentes parametros
| experimento | o que muda |
|---|---|
| exp1 | referencia: lr 2e-4, rank 16, 2 epocas |
| exp2 | learning rate menor (5e-5) |
| exp3 | LoRA menor (rank 4) |
| exp4 | so 1 epoca |
| exp5 | learning rate alto (1e-3) |

In [ ]:
!python src/experimentos.py --pular-base

## 6. Resultados

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("resultados/comparacao.md", encoding="utf-8").read()))
display(Image("resultados/loss_validacao.png"))
display(Image("resultados/acerto_total.png"))

In [ ]:
# Alguns exemplos de resposta do melhor modelo
import json, glob
melhor = max((json.load(open(f)) for f in glob.glob("resultados/avaliacao_exp*.json")), key=lambda r: r["acerto_total"])
print("Melhor:", melhor["nome"], melhor["acerto_total"], "%")
for l in open(f"resultados/predicoes_{melhor['nome']}.jsonl", encoding="utf-8").readlines()[:5]:
    r = json.loads(l)
    print()
    print("LOG:     ", r["entrada"])
    print("RESPOSTA:", r["resposta"])

## 7. Uso de GPU durante o treino

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu --format=csv

## 8. Baixar resultados e adaptadores (para usar no frontend local e subir no GitHub)

In [ ]:
# Junta resultados + historicos + adaptador do MELHOR modelo (os outros nao precisam ir)
import json, glob, os
melhor = max((json.load(open(f)) for f in glob.glob("resultados/avaliacao_exp*.json")), key=lambda r: r["acerto_total"])["nome"]
!zip -qr resultados_colab.zip resultados runs/*/historico.json runs/{melhor}/adaptador
print("Melhor modelo:", melhor, "| zip:", round(os.path.getsize("resultados_colab.zip") / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download("resultados_colab.zip")
except Exception as e:
    print("Download automatico indisponivel:", e)

## 9. (Opcional) Frontend rodando na GPU do Colab

In [ ]:
# Gera um link publico para testar o frontend na GPU. Para parar: botao de stop (quadrado).
!python app/app.py --share